In [1]:
import pandas as pd 
import numpy as np
import joblib
import seaborn as sns 
import matplotlib.pyplot as plt 

In [2]:
df = pd.read_csv("../data/processed_data.csv")

In [3]:
lr_model = joblib.load("../models/lead_time_model.pkl")
model_features = joblib.load("../models/model_features.pkl")

print(df.shape)
print(len(model_features), "Features expected by the model")

(10194, 31)
13 Features expected by the model


In [4]:
factories = pd.DataFrame({
    'Factory': ["Lot's O' Nuts", "Wicked Choccy's", "Sugar Shack", "Secret Factory", "The Other Factory"],
    'Factory Lat': [32.881893, 32.076176, 48.119140, 41.446333, 35.117500],
    'Factory Lon': [-111.768036, -81.088371, -96.181150, -90.565487, -89.971107]
})

factories

,Factory,Factory Lat,Factory Lon
0,Lot's O' Nuts,32.881893,-111.768036
1,Wicked Choccy's,32.076176,-81.088371
2,Sugar Shack,48.119140,-96.181150
3,Secret Factory,41.446333,-90.565487
4,The Other Factory,35.117500,-89.971107


In [5]:
def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371

    return c * r

In [6]:
def simulate_factory_options(row):
    results = []

    for _, factory_row in factories.iterrows():
        distance = haversine_distance(
            factory_row["Factory Lat"], factory_row["Factory Lon"],
            row["Customer Lat"], row["Customer Lon"]
        )

        input_dict = {col: 0 for col in model_features}
        input_dict["Distance (km)"] = distance

        ship_mode_col = "Ship Mode_" + row["Ship Mode"]
        if ship_mode_col in input_dict:
            input_dict[ship_mode_col] = 1

        factory_col = "Origin Factory_" + factory_row["Factory"]
        if factory_col in input_dict:
            input_dict[factory_col] = 1

        region_col = "Region_" + row["Region"]
        if region_col in input_dict:
            input_dict[region_col] = 1

        division_col = "Division_" + row["Division"]
        if division_col in input_dict:
            input_dict[division_col] = 1


        input_df = pd.DataFrame([input_dict])[model_features]
        predicted_lead_time = lr_model.predict(input_df)[0]

        results.append({
            "Factory": factory_row["Factory"],
            "Distance (km)": distance,
            "Predicted Lead Time": predicted_lead_time
        })

    return pd.DataFrame(results)

In [7]:
# test_order = df.iloc[0]
# simulate_factory_options(test_order)

In [8]:
unique_scenarios = df.drop_duplicates(subset= ["Origin Factory", "Region", "Ship Mode", "Division"])
print(len(unique_scenarios), "unique scenarios to simulate")

81 unique scenarios to simulate


In [9]:
all_results = []

for idx, scenario_row in unique_scenarios.iterrows():
    sim_result = simulate_factory_options(scenario_row)

    sim_result["Original Factory"] = scenario_row["Origin Factory"]
    sim_result["Region"] = scenario_row["Region"]
    sim_result["Ship Mode"] = scenario_row["Ship Mode"]
    sim_result["Division"] = scenario_row["Division"]

    all_results.append(sim_result)

simulation_results = pd.concat(all_results, ignore_index= True)
print(simulation_results.shape)
simulation_results.head(10)

(405, 7)


,Factory,Distance (km),Predicted Lead Time,Original Factory,Region,Ship Mode,Division
0,Lot's O' Nuts,1594.307024,5.522018,Wicked Choccy's,Interior,Standard Class,Chocolate
1,Wicked Choccy's,1385.223052,5.238815,Wicked Choccy's,Interior,Standard Class,Chocolate
2,Sugar Shack,2042.742314,6.129768,Wicked Choccy's,Interior,Standard Class,Chocolate
3,Secret Factory,1369.564071,5.222508,Wicked Choccy's,Interior,Standard Class,Chocolate
4,The Other Factory,781.695928,4.443104,Wicked Choccy's,Interior,Standard Class,Chocolate
5,Lot's O' Nuts,2300.484764,6.400455,Lot's O' Nuts,Interior,Standard Class,Chocolate
6,Wicked Choccy's,1246.456971,5.066200,Lot's O' Nuts,Interior,Standard Class,Chocolate
7,Sugar Shack,946.424484,4.766023,Lot's O' Nuts,Interior,Standard Class,Chocolate
8,Secret Factory,204.239322,3.772924,Lot's O' Nuts,Interior,Standard Class,Chocolate
9,The Other Factory,756.829994,4.412173,Lot's O' Nuts,Interior,Standard Class,Chocolate


In [10]:
best_factory_per_scenario = simulation_results.loc[
    simulation_results.groupby(["Original Factory", "Region", "Ship Mode", "Division"])["Predicted Lead Time"].idxmin()
]

best_factory_per_scenario.head(10)

,Factory,Distance (km),Predicted Lead Time,Original Factory,Region,Ship Mode,Division
153,Secret Factory,658.435230,2.342506,Lot's O' Nuts,Atlantic,First Class,Chocolate
211,Wicked Choccy's,1024.603225,1.823076,Lot's O' Nuts,Atlantic,Same Day,Chocolate
91,Wicked Choccy's,1024.603225,3.814069,Lot's O' Nuts,Atlantic,Second Class,Chocolate
58,Secret Factory,704.525630,4.407351,Lot's O' Nuts,Atlantic,Standard Class,Chocolate
81,Wicked Choccy's,373.940272,1.949218,Lot's O' Nuts,Gulf,First Class,Chocolate
156,Wicked Choccy's,214.060617,0.778592,Lot's O' Nuts,Gulf,Same Day,Chocolate
51,Wicked Choccy's,139.534895,2.676880,Lot's O' Nuts,Gulf,Second Class,Chocolate
19,The Other Factory,365.243531,3.900948,Lot's O' Nuts,Gulf,Standard Class,Chocolate
118,Secret Factory,399.176857,2.007900,Lot's O' Nuts,Interior,First Class,Chocolate
184,The Other Factory,681.118255,1.338733,Lot's O' Nuts,Interior,Same Day,Chocolate


In [11]:
current_performance = simulation_results[
    simulation_results["Factory"] == simulation_results["Original Factory"]    
][["Original Factory", "Region", "Ship Mode", "Division", "Predicted Lead Time"]].rename(
    columns = {"Predicted Lead Time": "Current Lead Time"}
)

recommendations = best_factory_per_scenario.merge(
    current_performance, on = ["Original Factory", "Region", "Ship Mode", "Division"]
)

recommendations["Improvement (days)"] = recommendations["Current Lead Time"] - recommendations["Predicted Lead Time"]
recommendations["Improvement (%)"] = (recommendations["Improvement (days)"]/ recommendations["Current Lead Time"]) * 100

recommendations = recommendations.sort_values("Improvement (days)", ascending = False)
recommendations.head(15)

,Factory,Distance (km),Predicted Lead Time,Original Factory,Region,Ship Mode,Division,Current Lead Time,Improvement (days),Improvement (%)
6,Wicked Choccy's,139.534895,2.676880,Lot's O' Nuts,Gulf,Second Class,Chocolate,6.219425,3.542545,56.959362
79,Lot's O' Nuts,614.438286,3.283115,Wicked Choccy's,Pacific,Second Class,Chocolate,6.787690,3.504575,51.631337
78,Lot's O' Nuts,614.438286,1.292122,Wicked Choccy's,Pacific,Same Day,Chocolate,4.796697,3.504575,73.062260
77,Lot's O' Nuts,69.656550,1.586197,Wicked Choccy's,Pacific,First Class,Chocolate,5.054638,3.468441,68.618972
80,Lot's O' Nuts,1107.579150,4.884814,Wicked Choccy's,Pacific,Standard Class,Chocolate,8.196896,3.312082,40.406538
5,Wicked Choccy's,214.060617,0.778592,Lot's O' Nuts,Gulf,Same Day,Chocolate,4.072714,3.294122,80.882723
2,Wicked Choccy's,1024.603225,3.814069,Lot's O' Nuts,Atlantic,Second Class,Chocolate,6.721710,2.907641,43.257463
1,Wicked Choccy's,1024.603225,1.823076,Lot's O' Nuts,Atlantic,Same Day,Chocolate,4.730717,2.907641,61.463014
40,Wicked Choccy's,201.983952,3.750678,Sugar Shack,Gulf,Standard Class,Sugar,6.473318,2.722640,42.059422
4,Wicked Choccy's,373.940272,1.949218,Lot's O' Nuts,Gulf,First Class,Chocolate,4.664732,2.715514,58.213714


In [12]:
total_scenarios = len(recommendations)
scenarios_with_improvement = (recommendations["Improvement (days)"] > 0.01).sum()
avg_improvement_days = recommendations["Improvement (days)"].mean()
avg_improvement_pct = recommendations[recommendations["Improvement (days)"] > 0.01]["Improvement (%)"].mean()


print(f"Total scenarios analyzed: {total_scenarios}")
print(f"Scenarios where reassignment helps: {scenarios_with_improvement} ({scenarios_with_improvement/total_scenarios * 100:.1f}%)")
print(f"Average improvement across ALL scenarios: {avg_improvement_days:.2f} days")
print(f"Average improvement among scenarios that benefits: {avg_improvement_pct:.1f}%")

Total scenarios analyzed: 81
Scenarios where reassignment helps: 63 (77.8%)
Average improvement across ALL scenarios: 1.24 days
Average improvement among scenarios that benefits: 33.7%


In [13]:
'''
recommendations.to_csv("../data/recommendations.csv", index= False)
simulation_results.to_csv("../data/simulation_results.csv", index= False)
'''

'\nrecommendations.to_csv("../data/recommendations.csv", index= False)\nsimulation_results.to_csv("../data/simulation_results.csv", index= False)\n'

In [14]:
kpi_lead_time_reduction = recommendations[recommendations["Improvement (days)"] > 0.01]["Improvement (%)"].mean()
print(f"KPI 1 - Avg Lead Time Reduction: {kpi_lead_time_reduction:.1f}%")

KPI 1 - Avg Lead Time Reduction: 33.7%


In [16]:
current_profit = df.groupby(["Origin Factory", "Region", "Division"])["Gross Profit"].mean().reset_index()
current_profit = current_profit.rename(columns= {"Gross Profit": "Current Avg Profit","Origin Factory": "Original Factory"})

profit_check = recommendations.merge(current_profit, on= ["Original Factory", "Region", "Division"], how= "left")

profit_check["Profit Impact"] = 0
kpi_profit_stability = "Stable - Distance/Factory reassignment showed ~0.01 correlation with Gross Profit (EDA heatmap)"
print(f"KPI 2 - Profit Impact Stability: {kpi_profit_stability}")

profit_stability_corr = df['Distance (km)'].corr(df['Gross Profit'])
print(f"Correlation between Distance and Gross Profit: {profit_stability_corr:.3f}")
print("Near-zero correlation confirms factory reassignment does not meaningfully affect profitability.")

KPI 2 - Profit Impact Stability: Stable - Distance/Factory reassignment showed ~0.01 correlation with Gross Profit (EDA heatmap)
Correlation between Distance and Gross Profit: 0.007
Near-zero correlation confirms factory reassignment does not meaningfully affect profitability.


In [18]:
base_confidence = 0.894

scenario_order_counts = df.groupby(["Origin Factory", "Region", "Ship Mode", "Division"]).size().reset_index(name= "Supporting Orders")
scenario_order_counts = scenario_order_counts.rename(columns={'Origin Factory': 'Original Factory'})
recommendations_with_confidence = recommendations.merge(
    scenario_order_counts, on = ["Original Factory", "Region", "Ship Mode", "Division"], how = "left"
)

recommendations_with_confidence["Confidence Score"] = np.where(
    recommendations_with_confidence["Supporting Orders"] >= 20,
    base_confidence,
    base_confidence * 0.85
)

recommendations_with_confidence[["Original Factory", "Region", "Ship Mode", "Division", "Supporting Orders", "Confidence Score"]].head(10)

,Original Factory,Region,Ship Mode,Division,Supporting Orders,Confidence Score
0,Lot's O' Nuts,Gulf,Second Class,Chocolate,183,0.8940
1,Wicked Choccy's,Pacific,Second Class,Chocolate,256,0.8940
2,Wicked Choccy's,Pacific,Same Day,Chocolate,76,0.8940
3,Wicked Choccy's,Pacific,First Class,Chocolate,195,0.8940
4,Wicked Choccy's,Pacific,Standard Class,Chocolate,815,0.8940
5,Lot's O' Nuts,Gulf,Same Day,Chocolate,49,0.8940
6,Lot's O' Nuts,Atlantic,Second Class,Chocolate,304,0.8940
7,Lot's O' Nuts,Atlantic,Same Day,Chocolate,89,0.8940
8,Sugar Shack,Gulf,Standard Class,Sugar,3,0.7599
9,Lot's O' Nuts,Gulf,First Class,Chocolate,126,0.8940


In [19]:
df['scenario_key'] = df['Origin Factory'] + '|' + df['Region'] + '|' + df['Ship Mode'] + '|' + df['Division']
recommendations['scenario_key'] = recommendations['Original Factory'] + '|' + recommendations['Region'] + '|' + recommendations['Ship Mode'] + '|' + recommendations['Division']

covered_orders = df['scenario_key'].isin(recommendations['scenario_key']).sum()
total_orders = len(df)
kpi_coverage = (covered_orders / total_orders) * 100

print(f"KPI 4 - Recommendation Coverage: {kpi_coverage:.1f}% ({covered_orders} of {total_orders} orders)")

KPI 4 - Recommendation Coverage: 100.0% (10194 of 10194 orders)


In [ ]:
'''
kpi_summary = pd.DataFrame({
    'KPI': ['Lead Time Reduction (%)', 'Profit Impact Stability', 'Scenario Confidence Score', 'Recommendation Coverage (%)'],
    'Value': [
        f"{kpi_lead_time_reduction:.1f}%",
        f"Stable (corr={profit_stability_corr:.3f})",
        f"{base_confidence:.3f} avg (adjusted for low-volume scenarios)",
        f"{kpi_coverage:.1f}%"
    ],
    'Description': [
        'Avg lead-time reduction among scenarios that benefit from reassignment',
        'Correlation between distance/factory and gross profit (near-zero = profit-neutral)',
        'Model reliability, adjusted down for scenarios with <20 supporting orders',
        '% of real orders covered by simulated scenarios'
    ]
})

recommendations_with_confidence.to_csv("../data/recommendations_final.csv", index=False)
kpi_summary.to_csv("../data/kpi_summary.csv", index=False)
kpi_summary
'''

,KPI,Value,Description
0,Lead Time Reduction (%),33.7%,Avg lead-time reduction among scenarios that b...
1,Profit Impact Stability,Stable (corr=0.007),Correlation between distance/factory and gross...
2,Scenario Confidence Score,0.894 avg (adjusted for low-volume scenarios),"Model reliability, adjusted down for scenarios..."
3,Recommendation Coverage (%),100.0%,% of real orders covered by simulated scenarios
